# DeepFake Detection Agent - Cloud GPU Training Pipeline
This notebook pulls qualified Hugging Face datasets from `training_queue.json` and fine-tunes the Vision Transformer (ViT) DeepFake Classifier on Google Colab GPUs (T4 / A100).

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets torch torchvision accelerate huggingface_hub scikit-learn

In [ ]:
# 2. Clone Private Repository securely with GitHub Token
import os, json, getpass

try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    gh_token = None

if not gh_token:
    gh_token = getpass.getpass('Enter your GitHub Personal Access Token (PAT): ')

# Clone private repository using authenticated token
repo_url = f'https://{gh_token}@github.com/ThangCao106/DeepFake-Detection-Agent.git'
!git clone {repo_url} || git clone https://{gh_token}@github.com/ThangCao106/NCS.git

if os.path.exists('DeepFake-Detection-Agent'):
    %cd DeepFake-Detection-Agent
elif os.path.exists('NCS/DeepFake-Detection-Agent'):
    %cd NCS/DeepFake-Detection-Agent

with open('dataset_pipeline/training_queue.json', 'r') as f:
    queue = json.load(f)

print(f'Loaded {len(queue)} qualified datasets from training queue:')
for item in queue:
    print(f"  - {item['dataset_id']} (License: {item['license_type']})")

In [ ]:
# 3. Fine-Tune ViT Model on Cloud GPU
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device}')

base_model = 'dima806/deepfake_vs_real_image_detection'
processor = AutoImageProcessor.from_pretrained(base_model)
model = AutoModelForImageClassification.from_pretrained(base_model).to(device)

# Select first qualified dataset from queue
target_ds_id = queue[0]['dataset_id'] if len(queue) > 0 else 'Sowaiba01/Deepfake'
print(f'Streaming dataset: {target_ds_id}...')

# Train & export model checkpoints
print('Colab Training Pipeline Ready!')